# Scaled Dot-Product Attention

AI & ML Foundations: Attention and Transformers · *Mathematics for AI and Machine Learning* by Fuheng Wu

softmax(Q K^T / sqrt(d)) V as a heatmap, and why the sqrt(d) keeps softmax from saturating.

Companion code for the [interactive demo](https://math4ai.org/companion/ml/attention/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(3)

# The demo's defaults: 6 tokens, head dimension d = 64
T, d = 6, 64
Q, K, V = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)

# ── Attention by hand ───────────────────────────────────────────────────────
scores = Q @ K.T / math.sqrt(d)
A = torch.softmax(scores, dim=-1)            # each row sums to 1
out = A @ V
print("row sums of the attention matrix:", [round(v, 6) for v in A.sum(-1).tolist()])
print("matches F.scaled_dot_product_attention:",
      torch.allclose(out, F.scaled_dot_product_attention(Q[None], K[None], V[None])[0], atol=1e-6))

# ── Why divide by sqrt(d): raw dot products have variance d ─────────────────
print("\n   d    var(q·k) raw   var after /sqrt(d)   mean max weight (raw)   (scaled)")
for dd in (4, 16, 64, 256):
    q, k = torch.randn(20000, dd), torch.randn(20000, dd)
    dots = (q * k).sum(-1)
    Qd, Kd = torch.randn(200, T, dd), torch.randn(200, T, dd)
    S = Qd @ Kd.transpose(-1, -2)
    raw = torch.softmax(S, -1).max(-1).values.mean().item()
    scaled = torch.softmax(S / math.sqrt(dd), -1).max(-1).values.mean().item()
    print(f"{dd:4d}    {dots.var().item():9.2f}       {(dots / math.sqrt(dd)).var().item():9.3f}             "
          f"{raw:.3f}                  {scaled:.3f}")
# Unscaled, softmax puts nearly all weight on one key and its gradient vanishes.

row sums of the attention matrix: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
matches F.scaled_dot_product_attention: True

   d    var(q·k) raw   var after /sqrt(d)   mean max weight (raw)   (scaled)
   4         3.97           0.992             0.582                  0.404
  16        16.01           1.001             0.779                  0.418
  64        64.04           1.001             0.892                  0.423
 256       255.83           0.999             0.941                  0.416
